# Etapa 3 — Rede profunda com PyTorch

**Inteligência Artificial · Ciência da Computação · PUC Minas · Trabalho Prático 3**

Esta configuração é uma candidata adicional para a posição de rede profunda. Ela tem quatro camadas ocultas com 128, 64, 32 e 16 neurônios, ativação ReLU e dropout de 15% após as três primeiras camadas. Cada ajuste percorre o treino por 120 épocas.

Abra no Colab e escolha **Ambiente de execução → Executar tudo**. Forneça apenas o **treino.csv oficial**. Se a sessão tiver GPU NVIDIA e um PyTorch compatível, o notebook usa CUDA. Uma sessão em CPU também pode executar o experimento. As métricas podem variar entre ambientes por diferenças numéricas entre dispositivos e bibliotecas.

In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys
import tempfile
from hashlib import sha256
from IPython.display import Markdown, display

candidatos = [
    Path("data/raw/treino.csv"),
    Path("treino.csv"),
    Path("/content/treino.csv"),
]
treino_csv = next((arquivo for arquivo in candidatos if arquivo.is_file()), None)
if treino_csv is None:
    try:
        from google.colab import files
    except ImportError as erro:
        raise FileNotFoundError(
            "Coloque o treino.csv oficial na pasta inicial ou em data/raw e execute novamente."
        ) from erro
    treino_csv = Path("data/raw/treino.csv")
    treino_csv.parent.mkdir(parents=True, exist_ok=True)
    print("Selecione o treino.csv oficial do Canvas.")
    files.upload_file(str(treino_csv))

treino_csv = treino_csv.resolve()
with treino_csv.open(encoding="utf-8-sig", newline="") as arquivo:
    cabecalho = next(csv.reader(arquivo), [])
obrigatorias = {"class", "protocol_type", "service", "flag", "src_bytes", "total_bytes"}
if not obrigatorias.issubset(cabecalho):
    raise ValueError(
        f"O arquivo em {treino_csv} não tem o cabeçalho do treino oficial. "
        "Substitua-o pelo treino.csv correto e execute novamente."
    )
trabalho = Path(tempfile.mkdtemp(prefix="ia-etapa3-deep-learning-"))
codigo_dados = trabalho / "dados_etapa2.py"
codigo_tecnicas = trabalho / "tecnicas_etapa2.py"
codigo_experimentos = trabalho / "avaliar_pytorch.py"
print("Base encontrada:", treino_csv.name)

In [ ]:
VERSOES = {
    "numpy": "2.5.4",
    "pandas": "3.0.1",
    "scipy": "1.18.1",
    "scikit-learn": "1.7.2",
    "imbalanced-learn": "0.14.1",
    "joblib": "1.6.0",
    "threadpoolctl": "3.8.0",
    "sklearn-compat": "0.1.6",
}

def consultar_versoes(ambiente):
    consulta = (
        "import importlib.metadata as m, json, sys; "
        "print(json.dumps({n: next((d.version for d in m.distributions(name=n)), None) "
        "for n in sys.argv[1:]}))"
    )
    resposta = subprocess.check_output(
        [sys.executable, "-c", consulta, *VERSOES], env=ambiente, text=True
    )
    return json.loads(resposta)

ambiente = os.environ.copy()
ambiente["PYTHONIOENCODING"] = "utf-8"
ambiente["PYTHONNOUSERSITE"] = "1"
ambiente["OPENBLAS_NUM_THREADS"] = "1"
ambiente["OMP_NUM_THREADS"] = "1"
if consultar_versoes(ambiente) != VERSOES:
    assinatura = sha256(json.dumps(VERSOES, sort_keys=True).encode()).hexdigest()[:12]
    bibliotecas = Path(tempfile.gettempdir()) / f"ia-etapa2-{sys.implementation.cache_tag}-{assinatura}"
    caminho_existente = ambiente.get("PYTHONPATH")
    ambiente["PYTHONPATH"] = os.pathsep.join(filter(None, (str(bibliotecas), caminho_existente)))
    if consultar_versoes(ambiente) != VERSOES:
        print("Instalando as versões dos experimentos. Aguarde a conclusão desta célula.", flush=True)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
             "--only-binary=:all:", "--upgrade", "--target", str(bibliotecas),
             *[f"{nome}=={versao}" for nome, versao in VERSOES.items()]],
            check=True, env={**os.environ, "PYTHONNOUSERSITE": "1"},
        )
versoes_ativas = consultar_versoes(ambiente)
if versoes_ativas != VERSOES:
    raise RuntimeError(f"As versões do ambiente não coincidiram com as previstas: {versoes_ativas}")
print("Python:", sys.version.split()[0])
print("Bibliotecas:", versoes_ativas)

In [ ]:
import shutil

DISPOSITIVO = "auto"  # Opções também disponíveis: "cpu" e "cuda".
ambiente["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
versao_referencia = "2.10.0"
codigo_sonda = (
    "import json, torch; "
    "print(json.dumps({'versao': str(torch.__version__), 'cuda': torch.cuda.is_available()}))"
)

def consultar_pytorch():
    sonda = subprocess.run(
        [sys.executable, "-c", codigo_sonda], env=ambiente, capture_output=True, text=True,
    )
    return json.loads(sonda.stdout) if sonda.returncode == 0 else None

if DISPOSITIVO not in ("auto", "cpu", "cuda"):
    raise ValueError("Selecione auto, cpu ou cuda.")
nvidia_smi = shutil.which("nvidia-smi")
gpu_presente = False
if DISPOSITIVO != "cpu" and nvidia_smi:
    consulta = subprocess.run([nvidia_smi, "-L"], capture_output=True, text=True)
    gpu_presente = consulta.returncode == 0 and "GPU " in consulta.stdout
torch_atual = consultar_pytorch()
if DISPOSITIVO == "cuda" and not gpu_presente and not (torch_atual and torch_atual["cuda"]):
    raise RuntimeError("Selecione um ambiente com GPU NVIDIA no Colab ou escolha cpu.")

instalar_cuda = gpu_presente and not (torch_atual and torch_atual["cuda"])
if torch_atual is None or instalar_cuda:
    variante = "cu126" if instalar_cuda else "cpu"
    biblioteca_torch = Path(tempfile.gettempdir()) / f"ia-pytorch-{sys.implementation.cache_tag}-{versao_referencia}-{variante}"
    ambiente["PYTHONPATH"] = os.pathsep.join(
        filter(None, (str(biblioteca_torch), ambiente.get("PYTHONPATH")))
    )
    torch_atual = consultar_pytorch()
    if torch_atual is None or (instalar_cuda and not torch_atual["cuda"]):
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--only-binary=:all:",
             "--upgrade", "--target", str(biblioteca_torch), "--index-url",
             f"https://download.pytorch.org/whl/{variante}", f"torch=={versao_referencia}"],
            check=True, env={**os.environ, "PYTHONNOUSERSITE": "1"},
        )
        torch_atual = consultar_pytorch()
    if torch_atual is None or (instalar_cuda and not torch_atual["cuda"]):
        raise RuntimeError("O PyTorch instalado não pôde usar o ambiente escolhido. Confira o driver ou escolha cpu.")
print("PyTorch:", torch_atual)

## Preparação e avaliação

Reutilizamos as mesmas cinco divisões estratificadas por grupo e o pré-processamento das duas MLPs já avaliadas. Partimos da versão C, retiramos duplicatas de mesmo rótulo no ajuste, imputamos ausências numéricas pela mediana, usamos uma categoria explícita para ausências nominais e aplicamos one-hot e Z-score numérico. O oversampling de uma anomalia para dois registros normais é aplicado apenas durante o ajuste.

Em cada lote de 512, a rede calcula a entropia cruzada binária a partir dos logits. A chamada **perda.backward()** retropropaga os gradientes e **adam.step()** atualiza os pesos. O Adam usa taxa inicial de 0,001 e decaimento de peso de 0,0001. O dropout fica ativo no treino. As previsões de validação usam a rede em modo de avaliação e limiar de 0,5 para anomalia.

Calculamos a média do F1-macro nas quatro rodadas iniciais e mostramos a quinta divisão à parte. A arquitetura foi fixada antes da execução. A quinta divisão já faz parte do protocolo de avaliação interna do projeto. A comparação com a rede profunda anterior mede duas configurações completas, incluindo diferenças de arquitetura e regularização.

[Treino e retropropagação no PyTorch](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html) · [Reprodutibilidade no PyTorch](https://docs.pytorch.org/docs/stable/notes/randomness.html)

In [ ]:
%%writefile "{codigo_dados}"
"""Prepara os dados e cria divisoes agrupadas para as etapas 2 e 3."""

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

ALVO = "class"
COLUNAS_EXCLUIDAS_EXPERIMENTOS = (
    "num_outbound_cmds", "tempo_resposta_analista_min", "tipo_alerta_soc", "score_ids_legado",
)
SEMENTE = 42
NOMINAIS = ["protocol_type", "service", "flag", "protocolo_cod"]
COPIAS = ["src_bytes_kb", "dst_bytes_kb", "same_srv_pct", "protocolo_cod"]


def carregar_treino(caminho: str | Path) -> pd.DataFrame:
    """Lê o CSV oficial e valida a coluna alvo."""
    caminho = Path(caminho)
    if not caminho.is_file():
        raise FileNotFoundError(f"Base oficial não encontrada: {caminho}")
    dados = pd.read_csv(caminho)
    if ALVO not in dados or dados.empty:
        raise ValueError(f"A base deve ser não vazia e conter {ALVO!r}.")
    if dados[ALVO].isna().any() or not set(dados[ALVO].unique()).issubset({"normal", "anomaly"}):
        raise ValueError("A coluna class deve conter somente normal ou anomaly, sem ausências.")
    return dados.copy()


def identificar_taxas(dados: pd.DataFrame) -> list[str]:
    """Retorna apenas frações ``*_rate``; ``*_pct`` usa escala percentual."""
    return [c for c in dados.select_dtypes(include="number") if c.lower().endswith("rate")]


def preparar_base(dados):
    entradas = dados.drop(columns=["class", *COLUNAS_EXCLUIDAS_EXPERIMENTOS])
    for coluna in NOMINAIS:
        entradas[coluna] = entradas[coluna].astype("string").str.strip().str.casefold().replace("", pd.NA)

    invalidos = {}
    for coluna in ("duration", "src_bytes", "dst_bytes"):
        invalidos[coluna] = entradas[coluna].lt(0)
    for coluna in identificar_taxas(entradas):
        invalidos[coluna] = entradas[coluna].notna() & ~entradas[coluna].between(0, 1)
    invalidos["src_bytes_kb"] = invalidos["src_bytes"] | entradas["src_bytes_kb"].lt(0)
    invalidos["dst_bytes_kb"] = invalidos["dst_bytes"] | entradas["dst_bytes_kb"].lt(0)
    invalidos["total_bytes"] = invalidos["src_bytes"] | invalidos["dst_bytes"] | entradas["total_bytes"].lt(0)
    percentual = entradas["same_srv_pct"]
    invalidos["same_srv_pct"] = invalidos["same_srv_rate"] | (percentual.notna() & ~percentual.between(0, 100))
    for coluna, mascara in invalidos.items():
        entradas[coluna] = entradas[coluna].mask(mascara)

    representacoes = {
        "A": entradas,
        "B": entradas.drop(columns=COPIAS),
        "C": entradas.drop(columns=[*COPIAS, "total_bytes"]),
    }
    assert [m.shape[1] for m in representacoes.values()] == [45, 41, 40]
    grupos = representacoes["C"].groupby(list(representacoes["C"]), dropna=False, sort=False).ngroup()
    assert grupos.notna().all()
    return representacoes, grupos


def escolher_linhas(x, y, indices, regra):
    if regra == "todas":
        return indices
    chaves = x.iloc[indices].assign(rotulo=y[indices])
    mascara = np.ones(len(indices), dtype=bool)
    if regra in {"sem_conflitos", "sem_ambos"}:
        grupos = chaves.groupby(list(x), dropna=False, sort=False)["rotulo"]
        mascara &= grupos.transform("nunique").to_numpy() == 1
    if regra in {"sem_duplicatas", "sem_ambos"}:
        mascara &= ~chaves.duplicated().to_numpy()
    return indices[mascara]


def separar_divisoes(representacoes, y, grupos):
    divisor = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
    particoes = list(divisor.split(representacoes["C"], y, grupos))
    reserva = particoes[-1][1]
    divisoes = []
    for fold, (treino, validacao) in enumerate(particoes, 1):
        if fold != 5:
            treino = np.setdiff1d(treino, reserva)
            assert set(grupos.iloc[treino]).isdisjoint(grupos.iloc[reserva])
            assert set(grupos.iloc[validacao]).isdisjoint(grupos.iloc[reserva])
        assert set(grupos.iloc[treino]).isdisjoint(grupos.iloc[validacao])
        divisoes.append((fold, treino, validacao))
    return divisoes


In [ ]:
%%writefile "{codigo_tecnicas}"
"""Transformacoes ajustadas apenas no treino de cada divisao."""

import numpy as np
import pandas as pd
from sklearn import config_context
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, RobustScaler, StandardScaler
from dados_etapa2 import NOMINAIS, SEMENTE, escolher_linhas


def transformar_numericas(treino, validacao, metodo, escala):
    treino = treino.to_numpy(dtype=np.float64, na_value=np.nan)
    validacao = validacao.to_numpy(dtype=np.float64, na_value=np.nan)
    # Uma entrada sem observacoes no ajuste recebe zero em ambas as parcelas.
    # Valores observados apenas na validacao nao participam do ajuste do KNN.
    if metodo == "knn":
        vazias = np.isnan(treino).all(axis=0)
        treino[:, vazias] = 0.0
        validacao[:, vazias] = 0.0
    if metodo == "mediana_indicador":
        tem_ausencia = np.isnan(treino).any(axis=0)
        indicadores_treino = np.isnan(treino[:, tem_ausencia])
        indicadores_val = np.isnan(validacao[:, tem_ausencia])
    if metodo == "knn":
        # Limita o custo de vizinhança a uma amostra sem rótulos da parte de treino.
        normalizador = StandardScaler().fit(treino)
        amostra = np.random.default_rng(SEMENTE).choice(len(treino), size=min(2000, len(treino)), replace=False)
        imputador = KNNImputer(n_neighbors=5, keep_empty_features=True)
        imputador.fit(normalizador.transform(treino[amostra]))
        with config_context(working_memory=48):
            treino = normalizador.inverse_transform(imputador.transform(normalizador.transform(treino)))
            validacao = normalizador.inverse_transform(imputador.transform(normalizador.transform(validacao)))
    else:
        estrategia = "mean" if metodo == "media" else "median"
        imputador = SimpleImputer(strategy=estrategia, keep_empty_features=True)
        treino, validacao = imputador.fit_transform(treino), imputador.transform(validacao)
    normalizadores = {"zscore": StandardScaler, "robusta": RobustScaler, "minmax": MinMaxScaler}
    if escala in normalizadores:
        normalizador = normalizadores[escala]()
        treino, validacao = normalizador.fit_transform(treino), normalizador.transform(validacao)
    if metodo == "mediana_indicador":
        treino = np.column_stack((treino, indicadores_treino))
        validacao = np.column_stack((validacao, indicadores_val))
    return treino, validacao


def transformar_categoricas(treino, validacao, metodo, codificacao):
    estrategia = "most_frequent" if metodo == "moda" else "constant"
    colunas = list(treino)
    colunas_treino, colunas_validacao = [], []
    for coluna in colunas:
        valores_treino = treino[[coluna]].to_numpy(dtype=object, na_value=np.nan)
        valores_validacao = validacao[[coluna]].to_numpy(dtype=object, na_value=np.nan)
        # A moda nao existe quando toda a coluna do ajuste esta ausente.
        # Nesse caso a categoria explicita mantem a posicao da coluna.
        estrategia_coluna = "constant" if pd.isna(valores_treino).all() else estrategia
        imputador = SimpleImputer(strategy=estrategia_coluna, fill_value="__ausente__")
        colunas_treino.append(imputador.fit_transform(valores_treino))
        colunas_validacao.append(imputador.transform(valores_validacao))
    treino = np.column_stack(colunas_treino)
    validacao = np.column_stack(colunas_validacao)
    if codificacao == "inteiros":
        codificador = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        return codificador.fit_transform(treino), codificador.transform(validacao)
    cod_treino, cod_val = [], []
    for i, coluna in enumerate(colunas):
        agrupar = codificacao == "onehot_raros" and coluna == "service"
        codificador = OneHotEncoder(
            sparse_output=False,
            min_frequency=20 if agrupar else None,
            handle_unknown="infrequent_if_exist" if agrupar else "ignore",
        )
        cod_treino.append(codificador.fit_transform(treino[:, i:i + 1]))
        cod_val.append(codificador.transform(validacao[:, i:i + 1]))
    return np.column_stack(cod_treino), np.column_stack(cod_val)


def transformar(x, y, indices, validacao, config):
    indices = escolher_linhas(x, y, indices, config.linhas)
    treino, validacao = x.iloc[indices], x.iloc[validacao]
    categoricas = [c for c in NOMINAIS if c in x]
    numericas = x.columns.drop(categoricas).tolist()
    num_treino, num_val = transformar_numericas(treino[numericas], validacao[numericas], config.numericos, config.escala)
    cat_treino, cat_val = transformar_categoricas(treino[categoricas], validacao[categoricas], config.nominais, config.codificacao)
    return np.column_stack((num_treino, cat_treino)), np.column_stack((num_val, cat_val)), y[indices]


In [ ]:
%%writefile "{codigo_experimentos}"
"""Avalia uma MLP profunda com retropropagação explícita nas divisões oficiais do treino."""

import json
from pathlib import Path
import sys
from time import perf_counter
from types import SimpleNamespace

from imblearn.over_sampling import RandomOverSampler
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import torch
from torch import nn

from dados_etapa2 import SEMENTE, carregar_treino, preparar_base, separar_divisoes
from tecnicas_etapa2 import transformar

PREPARACAO = SimpleNamespace(
    linhas="sem_duplicatas", numericos="mediana", nominais="ausencia",
    codificacao="onehot", escala="zscore",
)
CAMADAS = (128, 64, 32, 16)
DROPOUT = 0.15
CLASSES = ["normal", "anomaly"]
EPOCAS = 120
LOTE = 512


def criar_rede(n_entradas):
    camadas = []
    for indice, n_saidas in enumerate(CAMADAS):
        camadas.extend((nn.Linear(n_entradas, n_saidas), nn.ReLU()))
        if indice < len(CAMADAS) - 1:
            camadas.append(nn.Dropout(DROPOUT))
        n_entradas = n_saidas
    camadas.append(nn.Linear(n_entradas, 1))
    return nn.Sequential(*camadas)


def preparar_matrizes(x, y, treino, validacao, dispositivo):
    x_tr, x_val, y_tr = transformar(x, y, treino, validacao, PREPARACAO)
    x_tr = np.asarray(x_tr, dtype=np.float32, order="C")
    x_val = np.asarray(x_val, dtype=np.float32, order="C")
    if not (np.isfinite(x_tr).all() and np.isfinite(x_val).all()):
        raise ValueError("A transformação produziu valores não finitos.")
    amostrador = RandomOverSampler(sampling_strategy=0.5, random_state=SEMENTE)
    x_tr, y_tr = amostrador.fit_resample(x_tr, y_tr)
    return (
        torch.as_tensor(x_tr, device=dispositivo),
        torch.as_tensor(x_val, device=dispositivo),
        torch.as_tensor(y_tr, device=dispositivo, dtype=torch.float32),
        y[validacao],
    )


def avaliar(x, y, divisao, dispositivo):
    fold, treino, validacao = divisao
    x_tr, x_val, y_tr, y_val = preparar_matrizes(x, y, treino, validacao, dispositivo)
    torch.manual_seed(SEMENTE)
    embaralhador = torch.Generator(device=dispositivo).manual_seed(SEMENTE)
    modelo = criar_rede(x_tr.shape[1]).to(dispositivo)
    criterio = nn.BCEWithLogitsLoss()
    adam = torch.optim.Adam(modelo.parameters(), lr=0.001, weight_decay=0.0001)
    historico = []
    inicio = perf_counter()
    modelo.train()
    for epoca in range(1, EPOCAS + 1):
        indices = torch.randperm(len(y_tr), generator=embaralhador, device=dispositivo)
        perda_total = torch.zeros((), device=dispositivo)
        exemplos = 0
        for lote in indices.split(LOTE):
            adam.zero_grad(set_to_none=True)
            logits = modelo(x_tr[lote]).squeeze(1)
            perda = criterio(logits, y_tr[lote])
            perda.backward()  # retropropagação dos gradientes dos pesos e vieses
            adam.step()        # atualização dos parâmetros pelos gradientes
            perda_total += perda.detach() * len(lote)
            exemplos += len(lote)
        perda_media = (perda_total / exemplos).item()
        if exemplos != len(y_tr) or not np.isfinite(perda_media):
            raise RuntimeError(f"Treino inválido na divisão {fold}, época {epoca}.")
        historico.append({"fold": fold, "epoca": epoca, "perda_treino": perda_media})
        if epoca % 30 == 0:
            print(f"Divisão {fold} · época {epoca}/{EPOCAS} · perda {perda_media:.4f}", flush=True)
    tempo_ajuste = perf_counter() - inicio

    modelo.eval()
    with torch.inference_mode():
        # Logit >= 0 equivale à probabilidade de anomalia >= 0,5.
        previsto = (modelo(x_val).squeeze(1) >= 0).to(torch.int32).cpu().numpy()
    relatorio = classification_report(
        y_val, previsto, labels=[0, 1], target_names=CLASSES, output_dict=True, zero_division=0,
    )
    resultado = {
        "fold": fold, "epocas": len(historico), "linhas_ajuste": len(y_tr),
        "linhas_validacao": len(y_val), "colunas": x_tr.shape[1],
        "parametros": sum(p.numel() for p in modelo.parameters()),
        "f1_macro": relatorio["macro avg"]["f1-score"], "recall_anomaly": relatorio["anomaly"]["recall"],
        "precisao_anomaly": relatorio["anomaly"]["precision"], "segundos_ajuste": tempo_ajuste,
        "matriz": confusion_matrix(y_val, previsto, labels=[0, 1]).tolist(),
        "relatorio": relatorio,
    }
    print(f"Divisão {fold} · F1-macro {resultado['f1_macro']:.4f} · ajuste {tempo_ajuste:.1f} s", flush=True)
    return resultado, historico


def escolher_dispositivo(preferencia):
    if preferencia not in ("auto", "cpu", "cuda"):
        raise ValueError("O dispositivo deve ser auto, cpu ou cuda.")
    cuda_disponivel = preferencia != "cpu" and torch.cuda.is_available()
    if preferencia == "cuda" and not cuda_disponivel:
        raise RuntimeError("CUDA indisponível no PyTorch desta sessão. Selecione GPU ou use cpu.")
    return torch.device("cuda" if cuda_disponivel else "cpu")


def executar(arquivo, pasta, preferencia):
    torch.set_num_threads(2)
    torch.set_num_interop_threads(1)
    torch.use_deterministic_algorithms(True)
    dispositivo = escolher_dispositivo(preferencia)
    nome = torch.cuda.get_device_name(dispositivo) if dispositivo.type == "cuda" else "CPU"
    print(f"PyTorch {torch.__version__} · {nome} · CUDA {torch.version.cuda}", flush=True)
    dados = carregar_treino(arquivo)
    y = (dados["class"] == "anomaly").to_numpy(dtype=np.int32)
    representacoes, grupos = preparar_base(dados)
    divisoes = separar_divisoes(representacoes, dados["class"].to_numpy(), grupos)
    rodadas, historico = [], []
    for divisao in divisoes:
        resultado, perdas = avaliar(representacoes["C"], y, divisao, dispositivo)
        rodadas.append(resultado)
        historico.extend(perdas)

    tabela = pd.DataFrame(rodadas).drop(columns=["matriz", "relatorio"])
    tabela.to_csv(pasta / "pytorch_por_divisao.csv", index=False)
    pd.DataFrame(historico).to_csv(pasta / "pytorch_curvas_treino.csv", index=False)
    comparacao = tabela.iloc[:4]
    resultado = {
        "camadas": CAMADAS, "classes": CLASSES, "dispositivo": dispositivo.type,
        "nome_dispositivo": nome, "torch": torch.__version__, "cuda": torch.version.cuda,
        "medias": {
            "f1_macro": comparacao["f1_macro"].mean(), "desvio_f1": comparacao["f1_macro"].std(),
            "recall_anomaly": comparacao["recall_anomaly"].mean(),
            "precisao_anomaly": comparacao["precisao_anomaly"].mean(),
        },
        "rodadas": tabela.to_dict(orient="records"), "reserva": rodadas[4],
    }
    (pasta / "pytorch_resultados.json").write_text(json.dumps(resultado, ensure_ascii=False), encoding="utf-8")


if __name__ == "__main__":
    executar(Path(sys.argv[1]), Path(sys.argv[2]), sys.argv[3])

In [ ]:
subprocess.run(
    [sys.executable, "-u", str(codigo_experimentos), str(treino_csv), str(trabalho), DISPOSITIVO],
    check=True, env=ambiente,
)
profunda = json.loads((trabalho / "pytorch_resultados.json").read_text(encoding="utf-8"))

## Resultados da rede profunda

As métricas são calculadas ao final das 120 épocas. As tabelas incluem F1-macro, recall e precisão de anomaly e a matriz de confusão da quinta divisão.

In [ ]:
def tabela(colunas, linhas):
    linhas = [colunas, ["---"] * len(colunas), *linhas]
    display(Markdown("\n".join("| " + " | ".join(map(str, linha)) + " |" for linha in linhas)))




media = profunda["medias"]
reserva = profunda["reserva"]
print(f"Ambiente: {profunda['nome_dispositivo']} · PyTorch {profunda['torch']}")
print(f"Quatro rodadas · F1-macro médio {media['f1_macro']:.4f} ± {media['desvio_f1']:.4f}.")
tabela(
    ("Divisão", "Épocas", "F1-macro", "Recall de anomaly", "Precisão de anomaly", "Ajuste (s)"),
    [(r["fold"], r["epocas"], f"{r['f1_macro']:.4f}", f"{r['recall_anomaly']:.2%}",
      f"{r['precisao_anomaly']:.2%}", f"{r['segundos_ajuste']:.1f}") for r in profunda["rodadas"]],
)
print(f"Quinta divisão · {reserva['parametros']:,} parâmetros.")
tabela(
    ("Classe real", *(f"Previsto {c}" for c in profunda["classes"])),
    [(classe, *linha) for classe, linha in zip(profunda["classes"], reserva["matriz"])],
)
print("Métricas por divisão:", trabalho / "pytorch_por_divisao.csv")
print("Perda por época:", trabalho / "pytorch_curvas_treino.csv")

**Material da disciplina.** Rede Neural – Backpropagation. O exemplo foi adaptado para as características tabulares da competição.

**Apoio de IA.** Codex auxiliou na montagem e na validação do experimento.